# Q2 conditional action-risk quantile

One frozen fit, saved Q2 baseline, same-wallet held-action comparison. See `plan.json`.

In [1]:
import hashlib,json,os
from pathlib import Path
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
os.environ['MLFLOW_DISABLE_AGENT_HINT']='1'
import numpy as np
import pandas as pd
import qlib
from qlib.workflow import R
from src.core.recorder_sources import load_verified_recorder_objects
from src.core.native_evidence import prepare_groups
from src.workflow.stage1_policy_iteration_fit import prepare_critic_dataset
from src.workflow.stage1_policy_iteration_weights import prepare_label_weights
from src.model.stage1_conditional_action_risk import fit_conditional_action_risk
from src.analysis.stage1_policy_iteration_interpolation import _pinball
root=Path.cwd();card=root/'research/stage1_q2_conditional_action_risk_20261112'
plan=json.loads((card/'plan.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
qlib.init(provider_uri=str(root/'data/qlib/stage1_event_guard_fullminute_20261028'),region='us',kernels=1)
def source(card_key,id_key,hash_key,artifacts):
    path=root/plan['sources'][card_key]
    return load_verified_recorder_objects(path,expected_recorder_id=plan['sources'][id_key],artifacts=artifacts,expected_manifest_sha256=plan['sources'][hash_key])
labels,lr=source('labels_card','labels_recorder_id','labels_manifest_sha256',['full_label_panel'])
models,mr=source('saved_models_card','saved_models_recorder_id','saved_models_manifest_sha256',['risk_model'])
old,orr=source('old_gate_card','old_gate_recorder_id','old_gate_manifest_sha256',['daily_loss_improvements','report'])
old_model=models['risk_model'];panel=labels['full_label_panel']
dataset,audit=prepare_critic_dataset(panel,head='risk')
train=prepare_groups(dataset,'train')
z=train['feature'].to_numpy(float);u=train['action'].to_numpy(float)
y=train[('label','risk')].to_numpy(float);w=train[('weight','sample')].to_numpy(float)
state_raw=old_model.raw(z,np.zeros_like(u))
fit=fit_conditional_action_risk(z,u,y,state_raw,w,saved_scaler=old_model.scaler)
assert np.max(np.abs(fit.correction(z[:100],np.zeros_like(u[:100]))))<1e-12
frame=prepare_label_weights(panel);frame['day']=pd.to_datetime(frame.day).dt.normalize()
held=frame.loc[frame.held_physical_independent & frame.risk_known & frame.day.dt.year.isin([2019,2020])].copy()
assert len(held)==7485
hz=np.asarray(held.z.tolist(),float);hu=np.asarray(held.u.tolist(),float);actual=held.risk.to_numpy(float)
floor=hz[:,5];raw0=old_model.raw(hz,np.zeros_like(hu))
held['candidate_prediction']=fit.predict(hz,hu,raw0,floor)
held['baseline_prediction']=np.maximum(raw0,floor)
held['old_q2_prediction']=old_model.predict(hz,hu,floor)
held['improve_vs_baseline']=_pinball(actual,held.baseline_prediction.to_numpy())-_pinball(actual,held.candidate_prediction.to_numpy())
held['improve_vs_old_q2']=_pinball(actual,held.old_q2_prediction.to_numpy())-_pinball(actual,held.candidate_prediction.to_numpy())
cols=['improve_vs_baseline','improve_vs_old_q2']
calendar=old['daily_loss_improvements'].index
assert len(calendar)==505 and not calendar.has_duplicates
daily=held.groupby(['day','anchor_id'])[cols].mean().groupby(level='day').mean().reindex(calendar)
annual={str(year):{c:float(daily.loc[daily.index.year==year,c].mean()) for c in cols} for year in (2019,2020)}
support={str(year):int(held.loc[held.day.dt.year==year,'day'].nunique()) for year in (2019,2020)}
# Frozen same-anchor action ranking and 30pct feasible-set transmission.
def winner(g,col):
    m=g[col].min();return min(g.loc[g[col]<=m+1e-10,'action_id'].astype(str))
changes=[]
for (day,anchor),g in held.groupby(['day','anchor_id']):
    arg=winner(g,'candidate_prediction')!=winner(g,'old_q2_prediction')
    cand=set(g.loc[g.candidate_prediction<=.30,'action_id'].astype(str))
    oldset=set(g.loc[g.old_q2_prediction<=.30,'action_id'].astype(str))
    changes.append((day,arg or cand!=oldset))
change_days=pd.DataFrame(changes,columns=['day','changed']).groupby('day').changed.any()
transmission={str(year):int(change_days.loc[change_days.index.year==year].sum()) for year in (2019,2020)}
# 20-native-day circular resampling, preserving all unknown native positions.
x=daily[cols].to_numpy(float);known=np.isfinite(x);numeric=np.where(known,x,0.);N=len(x);B=20;count=int(np.ceil(N/B));rng=np.random.default_rng(20261112)
draws=np.full((10000,2),np.nan)
for start in range(0,10000,500):
    batch=min(500,10000-start);beg=rng.integers(0,N,size=(batch,count))
    ix=((beg[:,:,None]+np.arange(B))%N).reshape(batch,-1)[:,:N]
    den=known[ix].sum(axis=1);num=numeric[ix].sum(axis=1)
    np.divide(num,den,out=draws[start:start+batch],where=den>0)
undefined=np.isnan(draws).sum(axis=0).tolist()
lower=[None if undefined[i] else float(np.quantile(draws[:,i],.0125)) for i in range(2)]
gates={'support':all(v>=100 for v in support.values()),
       'annual':all(annual[str(y)][c]>0 for y in (2019,2020) for c in cols),
       'block':all(v is not None and v>0 for v in lower),
       'transmission':all(v>=20 for v in transmission.values())}
report={'status':'conditional_action_risk_information_pass' if all(gates.values()) else 'conditional_action_risk_fixed_gate_failed',
 'historical_research_passed':False,'new_model_fits':1,'new_native_accounts':0,
 'fit_audit':audit,'fit_solver_status':fit.solver_status,'fit_solver_iterations':fit.solver_iterations,
 'held_rows':len(held),'support_days':support,'annual_dayweighted_pinball_improvement':annual,
 'block':{'native_positions':N,'length_days':B,'resamples':10000,'seed':20261112,'alpha':.0125,'undefined':undefined,'lower':lower},
 'transmission_changed_days':transmission,'gates':gates,
 'scope':'reused 2018-2020 optimization held actions; not temporal OOS, actor, account or SOXX result'}
with R.start(experiment_name='stage1_q2_conditional_action_risk_20261112',uri=str(card/'qlib_recorder')):
    R.save_objects(frozen_plan=plan,conditional_risk_model=fit,report=report,daily_improvements=daily,held_predictions=held[['day','anchor_id','action_id','risk','candidate_prediction','baseline_prediction','old_q2_prediction']].copy(),source_readbacks={'labels':lr,'models':mr,'old_gate':orr})
    rec=R.get_recorder();rid,eid=rec.id,str(rec.experiment_id)
local=card/'qlib_recorder'/eid/rid
names=['frozen_plan','conditional_risk_model','report','daily_improvements','held_predictions','source_readbacks']
manifest={'recorder_id':rid,'entries':[{'path':(local/'artifacts'/name).relative_to(card).as_posix(),'sha256':sha(local/'artifacts'/name),'bytes':(local/'artifacts'/name).stat().st_size} for name in names]}
(card/'verification_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
summary={'experiment':plan['experiment'],'recorder_id':rid,'experiment_id':eid,'recorder_uri':str((card/'qlib_recorder').resolve()),
 'plan_sha256':sha(card/'plan.json'),'model_code_sha256':sha(root/'src/model/stage1_conditional_action_risk.py'),
 **report,'artifact_policy':{'policy':'slim_card_recorder_first','recorder_only_artifacts':names}}
(card/'summary.json').write_text(json.dumps(summary,indent=2,allow_nan=False)+'\n')
check,rr=load_verified_recorder_objects(card,expected_recorder_id=rid,artifacts=['report','daily_improvements'],expected_manifest_sha256=sha(card/'verification_manifest.json'))
assert check['report']==report
pd.testing.assert_frame_equal(check['daily_improvements'],daily)
(card/'readback_receipt.json').write_text(json.dumps({'recorder_id':rid,'manifest_sha256':sha(card/'verification_manifest.json'),'readback':rr},indent=2)+'\n')
print(json.dumps({'status':report['status'],'annual':annual,'lower':lower,'transmission':transmission,'gates':gates,'recorder_id':rid},indent=2))

ValueError: blocked: missing summary recorder_local_dir

## Conclusion

**No SOXX pass; fixed version closed.** The single frozen fit completed and saved Recorder `e8df7168ef0b4554a7803caad9b0a4d8`, but the local readback call in the same cell failed because the summary omitted `recorder_local_dir`. The metadata-only repair and scoped NAS readback passed without a second fit; see `repair_plan.json`, `readback_receipt.json`, `nas_scoped_readback.json`, and `terminal_receipt.json`.

The conditional action fit is algebraically the old Q2 joint objective with its optimal state coefficients fixed. It changed no saved minimum-risk action or 30% feasible set in either year. The 2019 pinball and pooled block gates failed. Detailed metrics and the equivalence attribution are in `conclusion.md` and `summary.json`. Do not rerun this closed notebook or promote it to an account.